# 1. Setup & Clone Repository
Clone the GitHub repository and install dependencies.

In [ ]:
!git clone https://github.com/10Unknownboy/Malaria-Parasite-Detector.git
%cd Malaria-Parasite-Detector
!pip install -r requirements_colab.txt

Cloning into 'Malaria-Parasite-Detector'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (49/49), done.
Receiving objects: 100% (62/62), 73.13 KiB | 4.88 MiB/s, done.
remote: Total 62 (delta 19), reused 54 (delta 11), pack-reused 0 (from 0)
Resolving deltas: 100% (19/19), done.
/content/Malaria-Parasite-Detector


# 2. Download Dataset via Kagglehub
Download the dataset without an API key and move it to the data directory.

In [ ]:
import kagglehub
import os
import shutil

# Download latest version
path = kagglehub.dataset_download("iarunava/cell-images-for-detecting-malaria")
print("Path to dataset files:", path)

# The downloaded path will contain a 'cell_images' folder, which contains 'Parasitized' and 'Uninfected'
source_dir = os.path.join(path, "cell_images")
if not os.path.exists(os.path.join(source_dir, "Parasitized")):
    # Sometimes kagglehub extracts it differently, check subfolders
    for root, dirs, files in os.walk(path):
        if "Parasitized" in dirs and "Uninfected" in dirs:
            source_dir = root
            break

# Create data directory if it doesn't exist
os.makedirs("data", exist_ok=True)

# Copy the folders
print("Copying Parasitized images...")
!cp -r "$source_dir/Parasitized" data/
print("Copying Uninfected images...")
!cp -r "$source_dir/Uninfected" data/
print("Dataset ready at data/")

100%|██████████| 675M/675M [00:16<00:00, 41.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/iarunava/cell-images-for-detecting-malaria/versions/1
Copying Parasitized images...
Copying Uninfected images...
Dataset ready at data/


# 3. Execute Training Pipeline
Run the local training orchestrator to train all models, evaluate them, and generate Grad-CAMs.

In [ ]:
!python main.py

🚀 Starting Local Training Pipeline

📱 Device: cuda

--- 1. Sanity Checks ---

  🔍  Sanity Check Report
  ✅  Dataset Size
       Found 27,558 images (expected ~27,558, tolerance ±5 %: [26,180 – 28,935])
  ✅  Class Balance
       Parasitized: 13,779 (50.0%)  |  Uninfected: 13,779 (50.0%)  |  Acceptable: 45%–55%
  ✅  Image Loading
       Successfully loaded 50 random sample images.
  ✅  Model Output Shape
       No models provided — skipped.
  ✅  All checks passed — ready for training!


--- 2. Data Preparation ---
  ✅ Data insights saved to: /content/Malaria-Parasite-Detector/results/data_insights.txt

  📊  Dataset Class Balance
  Total images       : 27,558
  Parasitized  (1)   : 13,779  (50.0%)
  Uninfected   (0)   : 13,779  (50.0%)


--- 3. Initialize Models ---
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 224MB/s]
Downloading: "https://download.pytorch.org/models/mob

# 4. Package and Download Results
Move all charts, graphs, and metrics into the `models/` folder, zip it, and download.

In [ ]:
import os
from google.colab import files

# Move results and reports into the models folder so they get zipped together
if os.path.exists("results"):
    !cp -r results models/
if os.path.exists("reports"):
    !cp -r reports models/

print("Compressing models directory...")
!zip -r /content/malaria_models_and_results.zip models/

print("Downloading zip file...")
files.download('/content/malaria_models_and_results.zip')

Compressing models directory...
  adding: models/ (stored 0%)
  adding: models/resnet18_best.pth (deflated 8%)
  adding: models/class_mapping.json (deflated 39%)
  adding: models/.gitkeep (stored 0%)
  adding: models/reports/ (stored 0%)
  adding: models/reports/project_report.md (deflated 52%)
  adding: models/model_config.json (deflated 59%)
  adding: models/best_model_report.json (deflated 38%)
  adding: models/model_comparison.csv (deflated 41%)
  adding: models/mobilenetv2_best.pth (deflated 6%)
  adding: models/resnet18_history.csv (deflated 55%)
  adding: models/simple_cnn_history.csv (deflated 55%)
  adding: models/simple_cnn_best.pth (deflated 6%)
  adding: models/training_metrics.json (deflated 71%)
  adding: models/mobilenetv2_history.csv (deflated 57%)
  adding: models/results/ (stored 0%)
  adding: models/results/roc_curves/ (stored 0%)
  adding: models/results/roc_curves/simple_cnn_roc.png (deflated 13%)
  adding: models/results/roc_curves/mobilenetv2_roc.png (deflated 13

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 5. Run Streamlit Dashboard
Use Ngrok to expose the Streamlit app to the public internet.

In [ ]:
import getpass
from pyngrok import ngrok, conf
import time

print("Enter your ngrok authtoken (get it for free at https://dashboard.ngrok.com/get-started/your-authtoken):")
authtoken = getpass.getpass()
conf.get_default().auth_token = authtoken

# Terminate open tunnels if they exist
ngrok.kill()

# Open a tunnel on the default Streamlit port (8501)
public_url = ngrok.connect(8501).public_url
print(f"\n✅ Streamlit Dashboard is starting! It will be live at: {public_url}\n")

# Run Streamlit in the background
!nohup streamlit run app.py > streamlit_logs.txt 2>&1 &
time.sleep(3)
print("Dashboard is running. Check the URL above!")

Enter your ngrok authtoken (get it for free at https://dashboard.ngrok.com/get-started/your-authtoken):
··········

✅ Streamlit Dashboard is starting! It will be live at: https://884b-34-6-219-207.ngrok-free.app

Dashboard is running. Check the URL above!
